# Multiple Linear Regression from scratch 
Covers: hyperplane hypothesis, normal equation, gradient descent, scaling (z-score / min-max),
one-hot encoding + dummy trap, correlation vs regression, multicollinearity (VIF),
Ridge, Adjusted R^2, F-statistic, t-statistics / p-values (computed without scipy).

In [ ]:
import os, math
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
os.makedirs('plots', exist_ok=True)
rng = np.random.default_rng(42)

## 1. Synthetic house-price data
Features: size (sq ft), bedrooms, bathrooms, location score, age, and a categorical "city".
Bedrooms are made correlated with size (multicollinearity).

In [ ]:
n = 200
size = rng.normal(2000, 500, n)
bedrooms = np.clip(np.round(size / 600 + rng.normal(0, 0.5, n)), 1, 6)
bathrooms = np.clip(np.round(bedrooms * 0.7 + rng.normal(0, 0.6, n)), 1, 5)
location = rng.uniform(1, 10, n)
age = rng.uniform(0, 40, n)
city = rng.choice(['A', 'B', 'C'], n)
city_eff = {'A': 0.0, 'B': 25.0, 'C': -15.0}
price = (50 + 0.12 * size + 8 * bathrooms + 12 * location - 1.5 * age
         + np.array([city_eff[c] for c in city]) + rng.normal(0, 20, n))   # $1000s

## 2. One-hot encoding with the dummy-variable trap (drop one column)

In [ ]:
def one_hot(values, drop_first=True):
    cats = sorted(set(values))
    if drop_first:
        cats = cats[1:]
    return np.array([[1.0 if v == c else 0.0 for c in cats] for v in values]), cats

D, cats = one_hot(city)
print("dummy columns (C-1):", cats)
D_full, _ = one_hot(city, drop_first=False)
Xtrap = np.c_[np.ones(n), D_full]
print("rank with all C dummies + intercept:", np.linalg.matrix_rank(Xtrap), "of", Xtrap.shape[1], "-> singular (trap)")

names = ['size', 'bedrooms', 'bathrooms', 'location', 'age'] + [f'city_{c}' for c in cats]
X = np.c_[size, bedrooms, bathrooms, location, age, D]
y = price

## 3. Scaling utilities and train/test split

In [ ]:
def zscore(X, mu=None, sd=None):
    if mu is None:
        mu, sd = X.mean(0), X.std(0)
    return (X - mu) / sd, mu, sd

def minmax(X):
    lo, hi = X.min(0), X.max(0)
    return (X - lo) / (hi - lo)

idx = rng.permutation(n)
tr, te = idx[:150], idx[150:]

## 4. Normal equation: beta = (X^T X)^-1 X^T Y, plus metrics

In [ ]:
def add_intercept(X):
    return np.c_[np.ones(len(X)), X]

def fit_normal(X, y, lam=0.0):
    """OLS / Ridge closed form; intercept (first column) is not penalised."""
    Xb = add_intercept(X)
    P = lam * np.eye(Xb.shape[1]); P[0, 0] = 0
    return np.linalg.solve(Xb.T @ Xb + P, Xb.T @ y)

def predict(X, beta):
    return add_intercept(X) @ beta

def r2(y, yp):
    return 1 - np.sum((y - yp) ** 2) / np.sum((y - y.mean()) ** 2)

def adj_r2(y, yp, k):
    """Adjusted R^2 = 1 - (1-R^2)(n-1)/(n-k-1)."""
    n = len(y)
    return 1 - (1 - r2(y, yp)) * (n - 1) / (n - k - 1)

def rmse(y, yp):
    return np.sqrt(np.mean((y - yp) ** 2))

beta = fit_normal(X[tr], y[tr])
yp_te = predict(X[te], beta)
print(f"Normal eq. test: R2={r2(y[te], yp_te):.3f}  adjR2={adj_r2(y[te], yp_te, X.shape[1]):.3f}  RMSE={rmse(y[te], yp_te):.2f}")
for nm, b in zip(['intercept'] + names, beta):
    print(f"  {nm:10s} {b:9.3f}")

## 5. Gradient descent on standardised features
Gradient of J = 1/(2m)||Xb - y||^2 is (1/m) X^T (Xb - y).

In [ ]:
def fit_gd(X, y, alpha=0.1, n_iters=500, lam=0.0):
    m, k = X.shape
    w, b = np.zeros(k), y.mean() * 0
    hist = []
    for _ in range(n_iters):
        err = X @ w + b - y
        w -= alpha * (X.T @ err / m + lam * w / m)
        b -= alpha * err.mean()
        hist.append(np.sum(err ** 2) / (2 * m))
    return w, b, hist

Xs_tr, mu, sd = zscore(X[tr])
w_gd, b_gd, hist = fit_gd(Xs_tr, y[tr], alpha=0.1, n_iters=500)
yp_gd = ((X[te] - mu) / sd) @ w_gd + b_gd
print(f"GD test R2={r2(y[te], yp_gd):.3f} (normal eq: {r2(y[te], yp_te):.3f})")

# GD raw (unscaled) vs scaled: convergence comparison
_, _, h_raw = fit_gd(X[tr] / X[tr].max(0), y[tr], alpha=0.1, n_iters=500)  # crude max scaling
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist, label='z-score'); ax[0].plot(h_raw, label='max-scaled')
ax[0].set_yscale('log'); ax[0].set_xlabel('iteration'); ax[0].set_ylabel('cost'); ax[0].legend()
ax[0].set_title('Gradient descent convergence')
# standardised coefficients = feature importance
order = np.argsort(np.abs(w_gd))
ax[1].barh(np.array(names)[order], w_gd[order]); ax[1].set_title('Standardised weights (importance)')
plt.tight_layout(); plt.savefig('plots/01_gd_and_importance.png', dpi=120); plt.close()

## 6. Correlation vs regression, and multicollinearity (VIF)
VIF_j = 1/(1-R^2_j), where R^2_j comes from regressing feature j on the others.

In [ ]:
corr = np.corrcoef(X[:, :5].T)
def vif(X):
    out = []
    for j in range(X.shape[1]):
        others = np.delete(X, j, axis=1)
        b = fit_normal(others, X[:, j])
        out.append(1 / (1 - r2(X[:, j], predict(others, b))))
    return np.array(out)

v = vif(X)
for nm, val in zip(names, v):
    print(f"VIF {nm:10s} {val:6.2f} {'<- high' if val > 5 else ''}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
im = ax[0].imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax[0].set_xticks(range(5)); ax[0].set_xticklabels(names[:5], rotation=45)
ax[0].set_yticks(range(5)); ax[0].set_yticklabels(names[:5]); plt.colorbar(im, ax=ax[0])
ax[0].set_title('Correlation matrix')
ax[1].bar(names, v); ax[1].axhline(5, c='orange', ls='--'); ax[1].axhline(10, c='r', ls='--')
ax[1].set_title('VIF'); ax[1].tick_params(axis='x', rotation=60)
plt.tight_layout(); plt.savefig('plots/02_correlation_vif.png', dpi=120); plt.close()

### Coefficient instability under near-duplicate features, fixed by Ridge

In [ ]:
x1 = rng.normal(size=100)
x2 = x1 + 0.02 * rng.normal(size=100)               # near-copy of x1
yy = 2 * x1 + rng.normal(0, 0.5, 100)
Xc = np.c_[x1, x2]
ols, ridge = [], []
for _ in range(200):
    i = rng.integers(0, 100, 100)                    # bootstrap resample
    ols.append(fit_normal(Xc[i], yy[i])[1:])
    ridge.append(fit_normal(Xc[i], yy[i], lam=5.0)[1:])
ols, ridge = np.array(ols), np.array(ridge)
print(f"OLS   coef std across bootstraps: {ols.std(0).round(2)}")
print(f"Ridge coef std across bootstraps: {ridge.std(0).round(2)}")
plt.figure(figsize=(5, 4))
plt.scatter(ols[:, 0], ols[:, 1], s=8, label='OLS'); plt.scatter(ridge[:, 0], ridge[:, 1], s=8, label='Ridge')
plt.xlabel('w1'); plt.ylabel('w2'); plt.legend(); plt.title('Bootstrap coefficients')
plt.tight_layout(); plt.savefig('plots/03_multicollinearity_ridge.png', dpi=120); plt.close()

## 7. Statistical tests: F-statistic and per-coefficient p-values
t_j = beta_j / SE_j, SE from sigma^2 (X^T X)^-1. p-values use the normal approximation to the
t-distribution (n is large); the F p-value is omitted (needs the F CDF) - a large F is what matters.

In [ ]:
def normal_sf(z):
    return 0.5 * math.erfc(z / math.sqrt(2))

def summary(X, y, names):
    Xb = add_intercept(X); n, p = Xb.shape
    beta = np.linalg.solve(Xb.T @ Xb, Xb.T @ y)
    resid = y - Xb @ beta
    sigma2 = resid @ resid / (n - p)
    se = np.sqrt(np.diag(sigma2 * np.linalg.inv(Xb.T @ Xb)))
    t = beta / se
    pvals = [2 * normal_sf(abs(z)) for z in t]
    R2 = r2(y, Xb @ beta); k = p - 1
    F = (R2 / k) / ((1 - R2) / (n - k - 1))
    print(f"R2={R2:.3f}  adjR2={adj_r2(y, Xb @ beta, k):.3f}  F={F:.1f} (df {k},{n-k-1})")
    for nm, b_, s_, t_, p_ in zip(['intercept'] + names, beta, se, t, pvals):
        print(f"  {nm:10s} coef={b_:8.3f} se={s_:7.3f} t={t_:7.2f} p={p_:.4f} {'*' if p_ < 0.05 else ''}")
summary(X[tr], y[tr], names)

## 8. R^2 vs Adjusted R^2 when adding junk features

In [ ]:
r2s, adjs, ks = [], [], []
Xtr_base = X[tr][:, :5]
for extra in range(0, 31, 3):
    junk = rng.normal(size=(len(tr), extra))
    Xa = np.c_[Xtr_base, junk]
    yp = predict(Xa, fit_normal(Xa, y[tr]))
    r2s.append(r2(y[tr], yp)); adjs.append(adj_r2(y[tr], yp, Xa.shape[1])); ks.append(extra)
plt.figure(figsize=(6, 4)); plt.plot(ks, r2s, 'o-', label='R2'); plt.plot(ks, adjs, 's-', label='Adjusted R2')
plt.xlabel('# junk features added'); plt.legend(); plt.title('Adjusted R2 penalises junk')
plt.tight_layout(); plt.savefig('plots/04_adjusted_r2.png', dpi=120); plt.close()
print(f"R2 {r2s[0]:.4f}->{r2s[-1]:.4f} (always up), adjR2 {adjs[0]:.4f}->{adjs[-1]:.4f}")

## 9. Scaling demo and Ridge path (test error vs lambda)

In [ ]:
print("MinMax range:", minmax(X[:, :5]).min().round(2), minmax(X[:, :5]).max().round(2))
Xs_tr, mu, sd = zscore(X[tr]); Xs_te = (X[te] - mu) / sd
lams = np.logspace(-2, 4, 25)
errs = [rmse(y[te], predict(Xs_te, fit_normal(Xs_tr, y[tr], l))) for l in lams]
plt.figure(figsize=(6, 4)); plt.semilogx(lams, errs); plt.xlabel('lambda'); plt.ylabel('test RMSE')
plt.title('Ridge: test error vs lambda'); plt.tight_layout(); plt.savefig('plots/05_ridge_path.png', dpi=120); plt.close()
print(f"best lambda={lams[int(np.argmin(errs))]:.2f}, RMSE={min(errs):.2f}")